In [2]:
import os
from typing import TypedDict, Annotated, Optional
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
llm = ChatOpenAI(model="gpt-4o-mini")

In [1]:
!uv add langgraph

Resolved 203 packages in 503ms                                       
Audited 190 packages in 41ms                                         


In [ ]:
# node : 함수(search, find ...)
# edge : node 들을 연결하는 연결선
# state : 그래프 전체에서 공유하는 dictionary

In [3]:
from langchain_core.runnables import RunnableLambda

In [4]:
chain = RunnableLambda(lambda x : x.upper()) | RunnableLambda(lambda x : x + "!")
print(chain.invoke('hello'))

HELLO!


In [9]:
class MiniState(TypedDict):
    text : str

def upper_node(state):
    return {"text" : state['text'].upper()}

def excite_node(state):
    return {"text" : state['text'] + '!'}

# 초기화 처리
b = StateGraph(MiniState)
b.add_node('upper', upper_node)
b.add_node('excite', excite_node)
b.add_edge(START, 'upper')
b.add_edge("upper", "excite") 
b.add_edge('excite', END)
b.compile().invoke({'text' : 'hello'})

{'text': 'HELLO!'}

In [12]:
class SimpleState(TypedDict):
    message: str

def greet(state: SimpleState) -> dict:
    return {"message" : f"안녕, {state['message']}"}

builder = StateGraph(SimpleState)
builder.add_node('greet', greet)
builder.add_edge(START, 'greet')
builder.add_edge('greet', END)
app = builder.compile()
app.invoke({'message' : '월요일'})

{'message': '안녕, 월요일'}

In [13]:
class NameState(TypedDict):
    name : str

def greet(state: NameState) -> dict:
    return {"name" : f"안녕하세요, {state['name']}님!"}

builder = StateGraph(NameState)
builder.add_node('greet', greet)
builder.add_edge(START, 'greet')
builder.add_edge('greet', END)
app = builder.compile()
app.invoke({'name' : '테스트'})

{'name': '안녕하세요, 테스트님!'}

In [14]:
class ChatSate(TypedDict):
    user_msg : str
    reply : str
    turn : int

class AnalaysisState(TypedDict):
    text : str
    word_count : int
    sentiment : str

b1 = StateGraph(ChatSate)
b2 = StateGraph(AnalaysisState)
type(b1).__name__, type(b2).__name__

('StateGraph', 'StateGraph')

In [15]:
# state에 들어가는 key값
class UserState(TypedDict):
    user_id : str
    name : str
    age : int
    is_active : bool

b1 = StateGraph(UserState)
UserState.__annotations__

{'user_id': str, 'name': str, 'age': int, 'is_active': bool}

In [16]:
# node 함수에 대한 규칙 - 변경된 key 값만 return 되도록
class CounterState(TypedDict):
    count : int
    note : str

def add_one(state : CounterState) -> dict:
    return {'count' : state['count'] + 1}

b = StateGraph(CounterState)
b.add_node('add_one', add_one)
b.add_edge(START, 'add_one')
b.add_edge('add_one', END)
b.compile().invoke({'count': 0, 'note' : '안녕하세요'})

{'count': 1, 'note': '안녕하세요'}

In [19]:
# multi-node serial
class TextState(TypedDict):
    text : str
    upper : str
    length : int

def to_upper(state: TextState) -> dict:
    return {"upper" : state['text'].upper()}

def measure(state: TextState) -> dict:
    return {"length" : len(state['text'])}

builder = StateGraph(TextState)
builder.add_node('upper', to_upper)
builder.add_node('measure', measure)

builder.add_edge(START, 'upper')
builder.add_edge('upper', 'measure')
builder.add_edge('measure', END)

app = builder.compile()
app.invoke({'text' : 'hello langgraph', 'upper': '', 'length': 0})

{'text': 'hello langgraph', 'upper': 'HELLO LANGGRAPH', 'length': 15}

In [20]:
class QAState(TypedDict):
    question : str
    answer : str

def llm_answer(state : QAState) -> dict:
    response = llm.invoke([HumanMessage(content=state['question'])])
    return {'answer' : response.content}

builder = StateGraph(QAState)
builder.add_node('llm_answer', llm_answer)

builder.add_edge(START, 'llm_answer')
builder.add_edge('llm_answer', END)

app = builder.compile()
result = app.invoke({'question' : 'langgraph를 한 문장으로?', 'answer': ''})

In [21]:
result

{'question': 'langgraph를 한 문장으로?',
 'answer': 'LangGraph는 언어 모델과 그래프 기반 구조를 결합하여 언어 데이터를 분석하고 시각적으로 표현하는 도구입니다.'}

In [31]:
# builder.add_conditional_edge
class NumberState(TypedDict):
    n : int
    label : str

def start_n(state):
    return {}

def even_handler(state):
    return {'label' : f"{state['n']}은 짝수입니다"}

def odd_handler(state):
    return {'label' : f"{state['n']}은 홀수입니다"}

# node 함수가 아닌 router
def parity_router(state) -> str:
    return "even" if state['n']%2 == 0 else "odd"

b = StateGraph(NumberState)
b.add_node('start_n', start_n)
b.add_node('even_handler', even_handler)
b.add_node('odd_handler', odd_handler)

b.add_edge(START, 'start_n')
b.add_conditional_edges('start_n', parity_router, {'even' : 'even_handler', 'odd' : 'odd_handler'})
b.add_edge('even_handler', END)
b.add_edge('odd_handler', END)
app = b.compile()

In [32]:
app.invoke({'n': 7, 'label' : ''})

{'n': 7, 'label': '7은 홀수입니다'}

In [33]:
app.invoke({'n': 8, 'label' : ''})

{'n': 8, 'label': '8은 짝수입니다'}

In [36]:
class GradeState(TypedDict):
    score : int
    grade : str

def start_grade(state): 
    return {}

def grade_a(state):
    return {'grade': 'A'}

def grade_b(state):
    return {'grade': 'B'}

def grade_c(state):
    return {'grade': 'C'}

def grade_router(state) -> str:
    if state['score'] >= 90: return 'a'
    if state['score'] >= 70: return 'b'
    return 'c'

b = StateGraph(GradeState)
b.add_node('start_grade', start_grade)
b.add_node('grade_a', grade_a)
b.add_node('grade_b', grade_b)
b.add_node('grade_c', grade_c)

b.add_edge(START, 'start_grade')
b.add_conditional_edges('start_grade', grade_router, {'a': 'grade_a', 'b': 'grade_b', 'c': 'grade_c'})

for n in ['grade_a', 'grade_b', 'grade_c']:
    b.add_edge(n, END)

app = b.compile()

In [37]:
class QuestionState(TypedDict):
    question : str
    qtype : str
    answer : str

def h_when(state):
    return {'answer' : '[시간] 답변'}
    
def h_where(state):
    return {'answer' : '[장소] 답변'}
    
def h_what(state):
    return {'answer' : '[일반] 답변'}

def h_stmt(state):
    return {'answer' : '[질술문] 검색 불필요'}

def merge(state):
    return {'answer' : state['answer'] + '!!!!!!'}

def classify(state):
    q = state['question'].lower()
    if '?' not in q: return {'qtype' : 'statement'}
    if any(w in q for w in ['언제', 'when']): return {'qtype' : 'when'}
    if any(w in q for w in ['어디', 'where']): return {'qtype' : 'where'}
    return {'qtype' : 'what'}

In [38]:
b = StateGraph(QuestionState)
b.add_node('classify', classify)
for name, fn in [('h_when', h_when), ('h_where', h_where), ('h_what', h_what), ('h_stmt', h_stmt)]:
    b.add_node(name, fn)
b.add_node('merge', merge)

b.add_edge(START, 'classify')
b.add_conditional_edges('classify', lambda s:s['qtype'], {'when': 'h_when', 'where': 'h_where', 'what': 'h_what', 'statement' : 'h_stmt'})

b.add_edge('h_when', 'merge')
b.add_edge('h_where', 'merge')
b.add_edge('h_what', 'merge')
b.add_edge('h_stmt', 'merge')
b.add_edge('merge', END)
app = b.compile()

In [39]:
app.invoke({'question' : '회의 어디서 해?', 'qtype': '', 'answer':''})

{'question': '회의 어디서 해?', 'qtype': 'where', 'answer': '[장소] 답변!!!!!!'}